# Comparing synthetic rating predictors

SVD, manual matrix factorization and neural networks on synthetic ratings.

Saved outputs are from the original coursework run, before the small fixes in this copy. The notebooks have not been rerun.


In [ ]:
# Libraries
import numpy as np
import pandas as pd
from surprise import SVD, Dataset, Reader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Concatenate

In [ ]:
# Parameters for synthetic dataset
num_users = 1000
num_items = 500
interaction_density = 0.05  # 5% of the user-item matrix will have interactions

# Generate synthetic user-item interactions
np.random.seed(42)
interactions = np.random.choice([0, 1], size=(num_users, num_items), p=[1-interaction_density, interaction_density])

## Generate synthetic ratings for interactions
ratings = interactions * np.random.randint(1, 6, (num_users, num_items))

## Adjusting user_ids and item_ids to start from 1
user_ids = np.repeat(range(1, num_users + 1), num_items)
item_ids = np.tile(range(1, num_items + 1), num_users)
ratings_flattened = ratings.flatten()

df = pd.DataFrame({'user_id': user_ids, 'item_id': item_ids, 'rating': ratings_flattened})
df = df[df['rating'] > 0]  # Keep only rows with interactions

## Split the data into training and testing sets
train, test = train_test_split(df, test_size=0.2, random_state=42)

# SVD (Collaborative Filtering)
reader = Reader(rating_scale=(1, 5))
train_data = Dataset.load_from_df(train[['user_id', 'item_id', 'rating']], reader)
trainset = train_data.build_full_trainset()

## Completed exercise
svd = SVD(random_state=42)
svd.fit(trainset)

## SVD Predictions
testset = list(zip(test['user_id'].values, test['item_id'].values, test['rating'].values))
svd_predictions = svd.test(testset)


### Convert predictions to a DataFrame for easier analysis
svd_preds_df = pd.DataFrame([(pred.uid, pred.iid, pred.r_ui, pred.est) for pred in svd_predictions],
                            columns=['user_id', 'item_id', 'actual_rating', 'svd_predicted_rating'])

## RMSE Calculation for SVD
actual_ratings = np.array([pred.r_ui for pred in svd_predictions])
svd_predicted_ratings = np.array([pred.est for pred in svd_predictions])
svd_rmse = np.sqrt(mean_squared_error(actual_ratings, svd_predicted_ratings))
print(f"SVD RMSE: {svd_rmse}")

SVD RMSE: 1.4691671472429109

In [ ]:
# Matrix Factorization with SGD

## Completed exercise - select the following parameters and complete the SGD code
n_latent_factors = 15
n_epochs = 10
alpha = 0.01

user_factors = np.random.normal(0, 1, (num_users, n_latent_factors))
item_factors = np.random.normal(0, 1, (num_items, n_latent_factors))

for epoch in range(n_epochs):
    for index, row in train.iterrows():
        # Adjusting for zero-based indexing
        user_i = int(row['user_id']) - 1
        item_i = int(row['item_id']) - 1
        rating = row['rating']
### Completed exercise - Apply SGD and complete the code
        error = rating - np.dot(user_factors[user_i], item_factors[item_i])
        user_factors[user_i] += alpha * (2 * error * item_factors[item_i])
        item_factors[item_i] += alpha * (2 * error * user_factors[user_i])

## Matrix Factorization Predictions
mf_predictions = [np.dot(user_factors[int(user)-1], item_factors[int(item)-1]) for (user, item, rating) in testset]
test['mf_predicted_rating'] = mf_predictions

## RMSE Calculation for Matrix Factorization with SGD
mf_rmse = np.sqrt(mean_squared_error(test['rating'], test['mf_predicted_rating']))
print(f"Matrix Factorization with SGD RMSE: {mf_rmse}")

# Neural Network-based Recommendation System

## Model 1

user_input = Input(shape=(1,))
item_input = Input(shape=(1,))
user_embedding = Embedding(num_users + 1, n_latent_factors)(user_input)
item_embedding = Embedding(num_items + 1, n_latent_factors)(item_input)

user_vector = Flatten()(user_embedding)
item_vector = Flatten()(item_embedding)

concat = Concatenate()([user_vector, item_vector])
fc1 = Dense(128, activation='relu')(concat)
fc2 = Dense(32, activation='relu')(fc1)
output = Dense(1)(fc2)

model = Model([user_input, item_input], output)
model.compile('adam', 'mean_squared_error')

X_train = [train.user_id.values, train.item_id.values]
Y_train = train.rating.values
model.fit(X_train, Y_train, epochs=5, verbose=1)

Matrix Factorization with SGD RMSE: 3.340240262516403

In [ ]:
## Model 1

## Completed exercise




### Neural Network Predictions
X_test = [test.user_id.values, test.item_id.values]
nn_predictions = model.predict(X_test)
test['nn_predicted_rating'] = nn_predictions.flatten()

### RMSE Calculation for Neural Network
nn_rmse = np.sqrt(mean_squared_error(test['rating'], test['nn_predicted_rating']))
print(f"Neural Network RMSE: {nn_rmse}")

### Merge SVD Predictions and Evaluate
test = test.drop(columns=[col for col in test.columns if col.startswith('svd_predicted_rating')]).merge(svd_preds_df[['user_id', 'item_id', 'svd_predicted_rating']], on=['user_id', 'item_id'], how='left')

# Display first few rows of predictions
print(test.head())

# End of Script
print("Recommendation systems models have been trained, predictions made, and RMSE calculated.")

Neural Network RMSE: 1.4549538118367196
   user_id  item_id  rating  mf_predicted_rating  nn_predicted_rating  \
0      975      302       4             7.144650             2.842202   
1      208      275       5             4.455099             3.006397   
2      812      262       1             1.530713             3.199834   
3      267       64       4            -7.690439             3.179863   
4      689       18       3             2.800346             2.951390   

   svd_predicted_rating_x  svd_predicted_rating_y  
0                2.743938                2.743938  
1                2.636277                2.636277  
2                3.031427                3.031427  
3                3.358686                3.358686  
4                3.276911                3.276911  
Recommendation systems models have been trained, predictions made, and RMSE calculated.

In [ ]:
# Lets play with the neural net and try adifferent optimizer
## Model 2

user_input = Input(shape=(1,))
item_input = Input(shape=(1,))
user_embedding = Embedding(num_users + 1, n_latent_factors)(user_input)
item_embedding = Embedding(num_items + 1, n_latent_factors)(item_input)

user_vector = Flatten()(user_embedding)
item_vector = Flatten()(item_embedding)

concat = Concatenate()([user_vector, item_vector])
fc1 = Dense(128, activation='sigmoid')(concat)
fc2 = Dense(32, activation='sigmoid')(fc1)
output = Dense(1)(fc2)

model = Model([user_input, item_input], output)
model.compile('SGD', 'mean_squared_error')

X_train = [train.user_id.values, train.item_id.values]
Y_train = train.rating.values
model.fit(X_train, Y_train, epochs=5, verbose=1)



### Neural Network Predictions
X_test = [test.user_id.values, test.item_id.values]
nn_predictions = model.predict(X_test)
test['nn_predicted_rating'] = nn_predictions.flatten()

### RMSE Calculation for Neural Network
nn_rmse = np.sqrt(mean_squared_error(test['rating'], test['nn_predicted_rating']))
print(f"Neural Network RMSE: {nn_rmse}")

### Merge SVD Predictions and Evaluate
test = test.drop(columns=[col for col in test.columns if col.startswith('svd_predicted_rating')]).merge(svd_preds_df[['user_id', 'item_id', 'svd_predicted_rating']], on=['user_id', 'item_id'], how='left')

# Display first few rows of predictions
print(test.head())

# End of Script
print("Recommendation systems models have been trained, predictions made, and RMSE calculated.")

Neural Network RMSE: 1.4525696926122018
   user_id  item_id  rating  mf_predicted_rating  nn_predicted_rating  \
0      975      302       4             7.144650             2.649182   
1      208      275       5             4.455099             2.605849   
2      812      262       1             1.530713             2.647527   
3      267       64       4            -7.690439             2.773491   
4      689       18       3             2.800346             2.788834   

   svd_predicted_rating_x  svd_predicted_rating_y  svd_predicted_rating_x  \
0                2.743938                2.743938                2.743938   
1                2.636277                2.636277                2.636277   
2                3.031427                3.031427                3.031427   
3                3.358686                3.358686                3.358686   
4                3.276911                3.276911                3.276911   

   svd_predicted_rating_y  
0                2.743938  
1 

## Original observation
The sigmoid/SGD configuration produced nearly constant predictions in the original run. This does not establish that sigmoid is unsuitable in general: both optimizer and activation changed.

In [ ]:
# Lets play with the neural net and try adifferentnumber of epochs
## Model 2

user_input = Input(shape=(1,))
item_input = Input(shape=(1,))
user_embedding = Embedding(num_users + 1, n_latent_factors)(user_input)
item_embedding = Embedding(num_items + 1, n_latent_factors)(item_input)

user_vector = Flatten()(user_embedding)
item_vector = Flatten()(item_embedding)

concat = Concatenate()([user_vector, item_vector])
fc1 = Dense(128, activation='relu')(concat)
fc2 = Dense(32, activation='relu')(fc1)
output = Dense(1)(fc2)

model = Model([user_input, item_input], output)
model.compile('SGD', 'mean_squared_error')

X_train = [train.user_id.values, train.item_id.values]
Y_train = train.rating.values
model.fit(X_train, Y_train, epochs=10, verbose=1)



### Neural Network Predictions
X_test = [test.user_id.values, test.item_id.values]
nn_predictions = model.predict(X_test)
test['nn_predicted_rating'] = nn_predictions.flatten()

### RMSE Calculation for Neural Network
nn_rmse = np.sqrt(mean_squared_error(test['rating'], test['nn_predicted_rating']))
print(f"Neural Network RMSE: {nn_rmse}")

### Merge SVD Predictions and Evaluate
test = test.drop(columns=[col for col in test.columns if col.startswith('svd_predicted_rating')]).merge(svd_preds_df[['user_id', 'item_id', 'svd_predicted_rating']], on=['user_id', 'item_id'], how='left')

# Display first few rows of predictions
print(test.head())

# End of Script
print("Recommendation systems models have been trained, predictions made, and RMSE calculated.")

Neural Network RMSE: 1.4652283012798464
   user_id  item_id  rating  mf_predicted_rating  nn_predicted_rating  \
0      975      302       4             7.144650             3.190702   
1      208      275       5             4.455099             2.870090   
2      812      262       1             1.530713             3.118284   
3      267       64       4            -7.690439             3.608766   
4      689       18       3             2.800346             3.466655   

   svd_predicted_rating_x  svd_predicted_rating_y  svd_predicted_rating_x  \
0                2.743938                2.743938                2.743938   
1                2.636277                2.636277                2.636277   
2                3.031427                3.031427                3.031427   
3                3.358686                3.358686                3.358686   
4                3.276911                3.276911                3.276911   

   svd_predicted_rating_y  svd_predicted_rating_x  svd_pre

# In this case the predictions seem to be a little bit more diverse (despite the fact that rmse has slighly inceased) cause the range seem to be a little bigger(with 5 epochs alla number returnes were close to 2.6 while with 10 is 3.35 for examle)

# Using the RMSE as a metric we observe that the NN and the SVD are doing equally good (both RMSE =1.46 ) while the MF has more than double (3.34)
(due to the lack of time i was not able to produse a full report but only some changes observations on the NN)

## Evaluation scope
The original ratings are independent random integers. A constant mean-rating baseline is essential before drawing conclusions about model quality. The historical RMSE observations above have not been rerun; they are not current benchmark results.